# MediMate Kidney Waitlist — Model Preparation

This notebook prepares the cleaned kidney waitlist dataset for machine learning.

Main goals:

- define the classification cohort
- prevent patient leakage
- create train / validation / test sets
- evaluate different CPRA handling strategies
- prepare numerical and categorical preprocessing
- create modeling-ready feature matrices

All preprocessing decisions are made carefully to avoid using future information.

In [5]:
# ============================================
# STEP 1 — Load cleaned modeling dataset
# ============================================

import pandas as pd
import numpy as np

from google.colab import files

# Upload the cleaned modeling dataset created
# in the Data Cleaning notebook
uploaded = files.upload()

Saving medimate_modeling_base.csv.gz to medimate_modeling_base.csv (1).gz


In [6]:
# ============================================
# STEP 2 — Load cleaned modeling dataset
# ============================================

model_df = pd.read_csv(
    "medimate_modeling_base.csv.gz",
    compression="gzip",
    parse_dates=["INIT_DATE"],
    low_memory=False
)

print("Dataset loaded successfully!")
print("Shape:", model_df.shape)

print("\nFirst 5 rows:")
display(model_df.head())

print("\nColumns:")
print(model_df.columns.tolist())

Dataset loaded successfully!
Shape: (494862, 52)

First 5 rows:


,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,END_CPRA,REM_CD,...,FUNC_STAT_TCR_clean,FUNC_STAT_invalid_flag,listing_year,INIT_CPRA_missing,A2A2B_status,BMI_missing,FUNC_STAT_missing,survival_valid,survival_time_days_clean,DAYSWAIT_CHRON_mismatch
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,0.0,8.0,...,2080.0,False,2020,0,B - missing,0,0,1,287.0,0
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,NaN,24.0,...,2070.0,False,2020,0,Not applicable (non-B),0,0,1,2322.0,0
2,N,NaN,F,O,32.85,2070.0,4099,0.0,0.0,8.0,...,2070.0,False,2020,0,Not applicable (non-B),0,0,1,604.0,0
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,0.0,13.0,...,2090.0,False,2020,0,Not applicable (non-B),0,0,1,909.0,0
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,0.0,13.0,...,2070.0,False,2020,0,Not applicable (non-B),0,0,1,231.0,0



Columns:
['ON_DIALYSIS', 'A2A2B_ELIGIBILITY', 'GENDER', 'ABO', 'BMI_TCR', 'FUNC_STAT_TCR', 'INIT_STAT', 'INIT_CPRA', 'END_CPRA', 'REM_CD', 'DAYSWAIT_CHRON', 'END_STAT', 'INIT_AGE', 'DIALYSIS_DATE', 'END_DATE', 'INIT_DATE', 'ETHCAT', 'PT_CODE', 'DAYSWAIT_ALLOC', 'COMPOSITE_DEATH_DATE', 'REGION', 'WL_ID_CODE', 'PREV_TX', 'TX_DATE', 'DON_TY', 'DIAG_KI', 'MULTIORG', 'ORGAN', 'PSTATUS', 'PTIME', 'TRR_ID_CODE', 'DONOR_ID', 'LISTING_CTR_CODE', 'outcome', 'event_adverse', 'event_transplant', 'censored', 'days_to_event', 'calculated_days_to_end', 'invalid_date_order', 'BMI_quality_flag', 'BMI_TCR_clean', 'FUNC_STAT_TCR_clean', 'FUNC_STAT_invalid_flag', 'listing_year', 'INIT_CPRA_missing', 'A2A2B_status', 'BMI_missing', 'FUNC_STAT_missing', 'survival_valid', 'survival_time_days_clean', 'DAYSWAIT_CHRON_mismatch']


In [7]:
# ============================================
# STEP 3 — Inspect classification cohort
# ============================================

print("OUTCOME COUNTS")
print(model_df["outcome"].value_counts())

print("\nEVENT_ADVERSE COUNTS")
print(model_df["event_adverse"].value_counts())

print("\nCENSORING COUNTS")
print(model_df["censored"].value_counts())

# Compare adverse-event label with censoring
print("\nADVERSE EVENT VS CENSORING")
print(
    pd.crosstab(
        model_df["event_adverse"],
        model_df["censored"],
        margins=True
    )
)

OUTCOME COUNTS
outcome
transplanted              234429
still_waiting             103300
removed_administrative     45332
transplanted_elsewhere     37243
removed_too_sick           35219
died                       34524
unknown                     4815
Name: count, dtype: int64

EVENT_ADVERSE COUNTS
event_adverse
0    425119
1     69743
Name: count, dtype: int64

CENSORING COUNTS
censored
0    391562
1    103300
Name: count, dtype: int64

ADVERSE EVENT VS CENSORING
censored            0       1     All
event_adverse                        
0              321819  103300  425119
1               69743       0   69743
All            391562  103300  494862


In [8]:
# ============================================
# STEP 4 — Inspect censoring by listing year
# ============================================

censoring_by_year = (
    model_df
    .groupby("listing_year")
    .agg(
        total_records=("WL_ID_CODE", "size"),
        still_waiting=("censored", "sum"),
        adverse_events=("event_adverse", "sum")
    )
)

# Percentage of each year's registrations
# that are still waiting
censoring_by_year["still_waiting_percent"] = (
    censoring_by_year["still_waiting"]
    / censoring_by_year["total_records"]
    * 100
).round(2)

print("CENSORING BY LISTING YEAR")
print(censoring_by_year)

CENSORING BY LISTING YEAR
              total_records  still_waiting  adverse_events  \
listing_year                                                 
2015                  36599            275            8321   
2016                  36956            432            7880   
2017                  37209            657            7529   
2018                  40564           1055            7915   
2019                  43223           1761            8306   
2020                  37825           2485            6841   
2021                  42130           4551            6637   
2022                  44442           7512            5880   
2023                  47003          12120            4809   
2024                  49635          19189            3632   
2025                  53231          31558            1819   
2026                  26045          21705             174   

              still_waiting_percent  
listing_year                         
2015                         

In [9]:
# ============================================
# STEP 5 — Check possible prediction horizons
# ============================================

horizons = {
    "6_months": 180,
    "1_year": 365,
    "2_years": 730
}

print("PREDICTION HORIZON FOLLOW-UP CHECK")

for name, days in horizons.items():

    # Still-waiting patients observed for at least
    # the full prediction horizon
    still_waiting_long_enough = (
        (model_df["censored"] == 1) &
        (model_df["survival_time_days_clean"] >= days)
    ).sum()

    # Still-waiting patients without enough follow-up
    still_waiting_too_short = (
        (model_df["censored"] == 1) &
        (model_df["survival_time_days_clean"] < days)
    ).sum()

    print(f"\n{name} ({days} days)")
    print("Still waiting with enough follow-up:", still_waiting_long_enough)
    print("Still waiting with insufficient follow-up:", still_waiting_too_short)

PREDICTION HORIZON FOLLOW-UP CHECK

6_months (180 days)
Still waiting with enough follow-up: 81680
Still waiting with insufficient follow-up: 21620

1_year (365 days)
Still waiting with enough follow-up: 64076
Still waiting with insufficient follow-up: 39224

2_years (730 days)
Still waiting with enough follow-up: 39695
Still waiting with insufficient follow-up: 63605


In [10]:
# ============================================
# STEP 6 — Inspect unknown outcomes
# ============================================

unknown_df = model_df[
    model_df["outcome"] == "unknown"
]

print("UNKNOWN OUTCOME RECORDS")
print("Count:", len(unknown_df))

print("\nAdverse-event labels:")
print(
    unknown_df["event_adverse"]
    .value_counts(dropna=False)
)

print("\nCensoring labels:")
print(
    unknown_df["censored"]
    .value_counts(dropna=False)
)

print("\nSurvival-time summary:")
print(
    unknown_df["survival_time_days_clean"]
    .describe()
)

UNKNOWN OUTCOME RECORDS
Count: 4815

Adverse-event labels:
event_adverse
0    4815
Name: count, dtype: int64

Censoring labels:
censored
0    4815
Name: count, dtype: int64

Survival-time summary:
count    4815.000000
mean      890.982762
std       750.118195
min         0.000000
25%       316.000000
50%       700.000000
75%      1288.500000
max      3899.000000
Name: survival_time_days_clean, dtype: float64


In [11]:
# ============================================
# STEP 7 — Create 1-year classification cohort
# ============================================

HORIZON_DAYS = 365

# Start with a copy
classification_df = model_df.copy()

# Initially, no row has an assigned 1-year target
classification_df["adverse_1yr"] = np.nan


# --------------------------------------------
# POSITIVE:
# adverse outcome occurred within 1 year
# --------------------------------------------
positive_mask = (
    (classification_df["survival_valid"] == 1) &
    (classification_df["event_adverse"] == 1) &
    (classification_df["survival_time_days_clean"] <= HORIZON_DAYS)
)

classification_df.loc[
    positive_mask,
    "adverse_1yr"
] = 1


# --------------------------------------------
# NEGATIVE:
# known non-adverse outcome
# --------------------------------------------
known_nonadverse_mask = (
    (classification_df["survival_valid"] == 1) &
    (classification_df["event_adverse"] == 0) &
    (classification_df["censored"] == 0) &
    (classification_df["outcome"] != "unknown")
)

classification_df.loc[
    known_nonadverse_mask,
    "adverse_1yr"
] = 0


# --------------------------------------------
# NEGATIVE:
# adverse event happened, but only AFTER 1 year
# --------------------------------------------
late_adverse_mask = (
    (classification_df["survival_valid"] == 1) &
    (classification_df["event_adverse"] == 1) &
    (classification_df["survival_time_days_clean"] > HORIZON_DAYS)
)

classification_df.loc[
    late_adverse_mask,
    "adverse_1yr"
] = 0


# --------------------------------------------
# NEGATIVE:
# still waiting but observed for at least 1 year
# --------------------------------------------
censored_enough_followup_mask = (
    (classification_df["survival_valid"] == 1) &
    (classification_df["censored"] == 1) &
    (classification_df["survival_time_days_clean"] >= HORIZON_DAYS)
)

classification_df.loc[
    censored_enough_followup_mask,
    "adverse_1yr"
] = 0


# Keep only rows where the 1-year outcome is known
classification_cohort = classification_df[
    classification_df["adverse_1yr"].notna()
].copy()

classification_cohort["adverse_1yr"] = (
    classification_cohort["adverse_1yr"].astype(int)
)


print("1-YEAR CLASSIFICATION COHORT")
print("Original rows:", len(model_df))
print("Eligible rows:", len(classification_cohort))
print("Excluded rows:", len(model_df) - len(classification_cohort))

print("\nTARGET COUNTS")
print(classification_cohort["adverse_1yr"].value_counts())

print("\nTARGET PERCENT")
print(
    classification_cohort["adverse_1yr"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

1-YEAR CLASSIFICATION COHORT
Original rows: 494862
Eligible rows: 450764
Excluded rows: 44098

TARGET COUNTS
adverse_1yr
0    433177
1     17587
Name: count, dtype: int64

TARGET PERCENT
adverse_1yr
0    96.1
1     3.9
Name: proportion, dtype: float64


In [12]:
# ============================================
# STEP 8 — Audit excluded classification rows
# ============================================

excluded_insufficient_followup = (
    (model_df["censored"] == 1) &
    (model_df["survival_time_days_clean"] < 365)
).sum()

excluded_unknown = (
    model_df["outcome"] == "unknown"
).sum()

excluded_invalid_survival = (
    model_df["survival_valid"] == 0
).sum()

print("CLASSIFICATION COHORT EXCLUSION AUDIT")

print(
    "Still waiting with < 1 year follow-up:",
    excluded_insufficient_followup
)

print(
    "Unknown outcomes:",
    excluded_unknown
)

print(
    "Invalid survival timelines:",
    excluded_invalid_survival
)

print(
    "\nTotal excluded:",
    excluded_insufficient_followup
    + excluded_unknown
    + excluded_invalid_survival
)

print(
    "Actual excluded:",
    len(model_df) - len(classification_cohort)
)

CLASSIFICATION COHORT EXCLUSION AUDIT
Still waiting with < 1 year follow-up: 39224
Unknown outcomes: 4815
Invalid survival timelines: 59

Total excluded: 44098
Actual excluded: 44098


In [13]:
# ============================================
# STEP 9 — Inspect classification cohort by year
# ============================================

cohort_by_year = (
    classification_cohort
    .groupby("listing_year")
    .agg(
        registrations=("WL_ID_CODE", "size"),
        unique_patients=("PT_CODE", "nunique"),
        adverse_1yr_events=("adverse_1yr", "sum")
    )
)

# Calculate 1-year adverse-event rate
cohort_by_year["adverse_1yr_percent"] = (
    cohort_by_year["adverse_1yr_events"]
    / cohort_by_year["registrations"]
    * 100
).round(2)

print("1-YEAR CLASSIFICATION COHORT BY YEAR")
print(cohort_by_year)

1-YEAR CLASSIFICATION COHORT BY YEAR
              registrations  unique_patients  adverse_1yr_events  \
listing_year                                                       
2015                  36009            34472                1452   
2016                  36412            34910                1408   
2017                  36716            35124                1383   
2018                  40097            38356                1351   
2019                  42705            40606                1568   
2020                  37368            35712                1640   
2021                  41672            39658                1760   
2022                  44003            41809                1675   
2023                  46608            44191                1691   
2024                  49323            46883                1907   
2025                  35543            33819                1580   
2026                   4308             4120                 172   

          

In [14]:
# ============================================
# STEP 10 — Check patient overlap across
# proposed temporal splits
# ============================================

train_candidate = classification_cohort[
    classification_cohort["listing_year"] <= 2022
]

val_candidate = classification_cohort[
    classification_cohort["listing_year"] == 2023
]

test_candidate = classification_cohort[
    classification_cohort["listing_year"] == 2024
]


# Unique patient IDs in each candidate set
train_patients = set(train_candidate["PT_CODE"])
val_patients = set(val_candidate["PT_CODE"])
test_patients = set(test_candidate["PT_CODE"])


print("PROPOSED TEMPORAL SPLIT")

print("\nTRAIN 2015-2022")
print("Registrations:", len(train_candidate))
print("Patients:", len(train_patients))

print("\nVALIDATION 2023")
print("Registrations:", len(val_candidate))
print("Patients:", len(val_patients))

print("\nTEST 2024")
print("Registrations:", len(test_candidate))
print("Patients:", len(test_patients))


# Check patient overlap
print("\nPATIENT OVERLAP")

print(
    "Train ↔ Validation:",
    len(train_patients & val_patients)
)

print(
    "Train ↔ Test:",
    len(train_patients & test_patients)
)

print(
    "Validation ↔ Test:",
    len(val_patients & test_patients)
)

PROPOSED TEMPORAL SPLIT

TRAIN 2015-2022
Registrations: 314982
Patients: 271769

VALIDATION 2023
Registrations: 46608
Patients: 44191

TEST 2024
Registrations: 49323
Patients: 46883

PATIENT OVERLAP
Train ↔ Validation: 6621
Train ↔ Test: 5340
Validation ↔ Test: 2294


In [15]:
# ============================================
# STEP 11 — Create patient-safe temporal splits
# ============================================

# -------------------------
# TRAIN: 2015–2022
# -------------------------
train_df = classification_cohort[
    classification_cohort["listing_year"] <= 2022
].copy()

train_patient_ids = set(train_df["PT_CODE"])


# -------------------------
# VALIDATION: 2023
# Only patients not seen in training
# -------------------------
val_df = classification_cohort[
    (classification_cohort["listing_year"] == 2023) &
    (~classification_cohort["PT_CODE"].isin(train_patient_ids))
].copy()

val_patient_ids = set(val_df["PT_CODE"])


# -------------------------
# TEST: 2024
# Only completely unseen patients
# -------------------------
seen_before_test = (
    train_patient_ids |
    val_patient_ids
)

test_df = classification_cohort[
    (classification_cohort["listing_year"] == 2024) &
    (~classification_cohort["PT_CODE"].isin(seen_before_test))
].copy()


print("PATIENT-SAFE TEMPORAL SPLIT")

print("\nTRAIN")
print("Rows:", len(train_df))
print("Patients:", train_df["PT_CODE"].nunique())

print("\nVALIDATION")
print("Rows:", len(val_df))
print("Patients:", val_df["PT_CODE"].nunique())

print("\nTEST")
print("Rows:", len(test_df))
print("Patients:", test_df["PT_CODE"].nunique())


# Final leakage check
train_ids = set(train_df["PT_CODE"])
val_ids = set(val_df["PT_CODE"])
test_ids = set(test_df["PT_CODE"])

print("\nPATIENT OVERLAP AFTER CLEANING")
print("Train ↔ Validation:", len(train_ids & val_ids))
print("Train ↔ Test:", len(train_ids & test_ids))
print("Validation ↔ Test:", len(val_ids & test_ids))

PATIENT-SAFE TEMPORAL SPLIT

TRAIN
Rows: 314982
Patients: 271769

VALIDATION
Rows: 39402
Patients: 37570

TEST
Rows: 41471
Patients: 39663

PATIENT OVERLAP AFTER CLEANING
Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0


In [16]:
# ============================================
# STEP 12 — Check target balance across splits
# ============================================

def show_target_balance(name, df):

    counts = df["adverse_1yr"].value_counts().sort_index()

    percentages = (
        df["adverse_1yr"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Target counts:")
    print(counts)

    print("Target percentages:")
    print(percentages)


print("TARGET BALANCE ACROSS SPLITS")

show_target_balance("TRAIN", train_df)
show_target_balance("VALIDATION", val_df)
show_target_balance("TEST", test_df)

TARGET BALANCE ACROSS SPLITS

TRAIN
Rows: 314982
Target counts:
adverse_1yr
0    302745
1     12237
Name: count, dtype: int64
Target percentages:
adverse_1yr
0    96.12
1     3.88
Name: proportion, dtype: float64

VALIDATION
Rows: 39402
Target counts:
adverse_1yr
0    37956
1     1446
Name: count, dtype: int64
Target percentages:
adverse_1yr
0    96.33
1     3.67
Name: proportion, dtype: float64

TEST
Rows: 41471
Target counts:
adverse_1yr
0    39866
1     1605
Name: count, dtype: int64
Target percentages:
adverse_1yr
0    96.13
1     3.87
Name: proportion, dtype: float64


In [17]:
# ============================================
# STEP 13 — Check CPRA missingness by split
# ============================================

def show_cpra_missingness(name, df):

    missing_count = df["INIT_CPRA"].isna().sum()

    missing_percent = (
        df["INIT_CPRA"].isna().mean() * 100
    )

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Missing INIT_CPRA:", missing_count)
    print(
        "Missing percent:",
        round(missing_percent, 2),
        "%"
    )


print("CPRA MISSINGNESS ACROSS SPLITS")

show_cpra_missingness("TRAIN", train_df)
show_cpra_missingness("VALIDATION", val_df)
show_cpra_missingness("TEST", test_df)

CPRA MISSINGNESS ACROSS SPLITS

TRAIN
Rows: 314982
Missing INIT_CPRA: 0
Missing percent: 0.0 %

VALIDATION
Rows: 39402
Missing INIT_CPRA: 31642
Missing percent: 80.31 %

TEST
Rows: 41471
Missing INIT_CPRA: 34658
Missing percent: 83.57 %


In [18]:
# ============================================
# STEP 14 — Inspect observed CPRA distribution
# ============================================

def show_cpra_distribution(name, df):

    observed = df["INIT_CPRA"].dropna()

    print(f"\n{name}")
    print("Observed CPRA values:", len(observed))

    print(
        observed.describe(
            percentiles=[0.25, 0.50, 0.75]
        )
    )


print("OBSERVED CPRA DISTRIBUTION")

show_cpra_distribution("TRAIN", train_df)
show_cpra_distribution("VALIDATION", val_df)
show_cpra_distribution("TEST", test_df)

OBSERVED CPRA DISTRIBUTION

TRAIN
Observed CPRA values: 314982
count    314982.000000
mean          6.499024
std          20.931179
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         100.000000
Name: INIT_CPRA, dtype: float64

VALIDATION
Observed CPRA values: 7760
count    7760.000000
mean       32.181137
std        34.390750
min         0.000000
25%         0.040000
50%        19.790000
75%        58.230000
max       100.000000
Name: INIT_CPRA, dtype: float64

TEST
Observed CPRA values: 6813
count    6813.000000
mean       43.040357
std        33.266271
min         0.000000
25%        11.420000
50%        37.380000
75%        71.260000
max       100.000000
Name: INIT_CPRA, dtype: float64


In [19]:
# ============================================
# STEP 15 — Check zero-valued CPRA by split
# ============================================

def show_cpra_zero_rate(name, df):

    observed = df["INIT_CPRA"].dropna()

    zero_count = (observed == 0).sum()

    zero_percent = (
        zero_count / len(observed) * 100
        if len(observed) > 0 else np.nan
    )

    print(f"\n{name}")
    print("Observed CPRA:", len(observed))
    print("CPRA = 0:", zero_count)
    print(
        "Percent of observed CPRA equal to 0:",
        round(zero_percent, 2),
        "%"
    )


print("ZERO-VALUED CPRA BY SPLIT")

show_cpra_zero_rate("TRAIN", train_df)
show_cpra_zero_rate("VALIDATION", val_df)
show_cpra_zero_rate("TEST", test_df)

ZERO-VALUED CPRA BY SPLIT

TRAIN
Observed CPRA: 314982
CPRA = 0: 271670
Percent of observed CPRA equal to 0: 86.25 %

VALIDATION
Observed CPRA: 7760
CPRA = 0: 1843
Percent of observed CPRA equal to 0: 23.75 %

TEST
Observed CPRA: 6813
CPRA = 0: 34
Percent of observed CPRA equal to 0: 0.5 %


In [20]:
# ============================================
# STEP 16 — Compare outcome by CPRA missingness
# ============================================

def compare_cpra_missingness_outcome(name, df):

    summary = (
        df
        .groupby("INIT_CPRA_missing")
        .agg(
            records=("WL_ID_CODE", "size"),
            adverse_events=("adverse_1yr", "sum")
        )
    )

    summary["adverse_percent"] = (
        summary["adverse_events"]
        / summary["records"]
        * 100
    ).round(2)

    print(f"\n{name}")
    print(summary)


print("1-YEAR ADVERSE RATE BY CPRA MISSINGNESS")

compare_cpra_missingness_outcome(
    "VALIDATION — 2023",
    val_df
)

compare_cpra_missingness_outcome(
    "TEST — 2024",
    test_df
)

1-YEAR ADVERSE RATE BY CPRA MISSINGNESS

VALIDATION — 2023
                   records  adverse_events  adverse_percent
INIT_CPRA_missing                                          
0                     7760             268             3.45
1                    31642            1178             3.72

TEST — 2024
                   records  adverse_events  adverse_percent
INIT_CPRA_missing                                          
0                     6813             273             4.01
1                    34658            1332             3.84


In [21]:
# ============================================
# STEP 17 — Final baseline feature set
# Exclude CPRA from primary model
# ============================================

baseline_features_no_cpra = [
    "INIT_AGE",
    "GENDER",
    "ETHCAT",
    "ABO",
    "ON_DIALYSIS",
    "BMI_TCR_clean",
    "BMI_missing",
    "FUNC_STAT_TCR_clean",
    "FUNC_STAT_missing",
    "INIT_STAT",
    "REGION",
    "A2A2B_status"
]

print("PRIMARY BASELINE FEATURES")
print("Number of features:", len(baseline_features_no_cpra))

for feature in baseline_features_no_cpra:
    print("✓", feature)

PRIMARY BASELINE FEATURES
Number of features: 12
✓ INIT_AGE
✓ GENDER
✓ ETHCAT
✓ ABO
✓ ON_DIALYSIS
✓ BMI_TCR_clean
✓ BMI_missing
✓ FUNC_STAT_TCR_clean
✓ FUNC_STAT_missing
✓ INIT_STAT
✓ REGION
✓ A2A2B_status


In [22]:
# ============================================
# STEP 18 — CPRA sensitivity feature set
# ============================================

features_with_cpra = [
    "INIT_AGE",
    "GENDER",
    "ETHCAT",
    "ABO",
    "INIT_CPRA",
    "INIT_CPRA_missing",
    "ON_DIALYSIS",
    "BMI_TCR_clean",
    "BMI_missing",
    "FUNC_STAT_TCR_clean",
    "FUNC_STAT_missing",
    "INIT_STAT",
    "REGION",
    "A2A2B_status"
]

print("CPRA SENSITIVITY FEATURE SET")
print("Number of features:", len(features_with_cpra))

for feature in features_with_cpra:
    print("✓", feature)

CPRA SENSITIVITY FEATURE SET
Number of features: 14
✓ INIT_AGE
✓ GENDER
✓ ETHCAT
✓ ABO
✓ INIT_CPRA
✓ INIT_CPRA_missing
✓ ON_DIALYSIS
✓ BMI_TCR_clean
✓ BMI_missing
✓ FUNC_STAT_TCR_clean
✓ FUNC_STAT_missing
✓ INIT_STAT
✓ REGION
✓ A2A2B_status


In [23]:
# ============================================
# STEP 19 — Define preprocessing feature types
# ============================================

# True continuous numerical features
numeric_features = [
    "INIT_AGE",
    "BMI_TCR_clean"
]

# Categorical features
# Some contain numeric codes, but those numbers
# represent categories rather than quantities.
categorical_features = [
    "GENDER",
    "ETHCAT",
    "ABO",
    "ON_DIALYSIS",
    "FUNC_STAT_TCR_clean",
    "INIT_STAT",
    "REGION",
    "A2A2B_status"
]

# Binary 0/1 indicators
binary_features = [
    "BMI_missing",
    "FUNC_STAT_missing"
]


print("FEATURE TYPE GROUPS")

print("\nNUMERICAL")
for col in numeric_features:
    print("✓", col)

print("\nCATEGORICAL")
for col in categorical_features:
    print("✓", col)

print("\nBINARY")
for col in binary_features:
    print("✓", col)

print(
    "\nTotal features:",
    len(numeric_features)
    + len(categorical_features)
    + len(binary_features)
)

print(
    "Matches primary feature set:",
    set(
        numeric_features
        + categorical_features
        + binary_features
    ) == set(baseline_features_no_cpra)
)

FEATURE TYPE GROUPS

NUMERICAL
✓ INIT_AGE
✓ BMI_TCR_clean

CATEGORICAL
✓ GENDER
✓ ETHCAT
✓ ABO
✓ ON_DIALYSIS
✓ FUNC_STAT_TCR_clean
✓ INIT_STAT
✓ REGION
✓ A2A2B_status

BINARY
✓ BMI_missing
✓ FUNC_STAT_missing

Total features: 12
Matches primary feature set: True


In [24]:
# ============================================
# STEP 20 — Check training feature missingness
# ============================================

train_missingness = (
    train_df[baseline_features_no_cpra]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

train_missingness_percent = (
    train_df[baseline_features_no_cpra]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

missing_summary = pd.DataFrame({
    "missing_count": train_missingness,
    "missing_percent": train_missingness_percent
})

print("TRAINING FEATURE MISSINGNESS")
print(missing_summary)

TRAINING FEATURE MISSINGNESS
                     missing_count  missing_percent
A2A2B_status                     0             0.00
ABO                              0             0.00
BMI_TCR_clean                 1123             0.36
BMI_missing                      0             0.00
ETHCAT                           0             0.00
FUNC_STAT_TCR_clean            220             0.07
FUNC_STAT_missing                0             0.00
GENDER                           0             0.00
INIT_AGE                         0             0.00
INIT_STAT                        0             0.00
ON_DIALYSIS                      0             0.00
REGION                           0             0.00


In [25]:
# ============================================
# STEP 21 — Check training-only BMI median
# ============================================

train_bmi_median = train_df["BMI_TCR_clean"].median()

print("TRAINING-ONLY BMI IMPUTATION VALUE")
print("BMI median:", train_bmi_median)

print(
    "\nMissing BMI values that would be imputed:",
    train_df["BMI_TCR_clean"].isna().sum()
)

TRAINING-ONLY BMI IMPUTATION VALUE
BMI median: 28.46

Missing BMI values that would be imputed: 1123


In [26]:
# ============================================
# STEP 22 — Separate features and target
# ============================================

X_train = train_df[baseline_features_no_cpra].copy()
y_train = train_df["adverse_1yr"].copy()

X_val = val_df[baseline_features_no_cpra].copy()
y_val = val_df["adverse_1yr"].copy()

X_test = test_df[baseline_features_no_cpra].copy()
y_test = test_df["adverse_1yr"].copy()


print("FEATURE / TARGET SHAPES")

print("\nTRAIN")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nVALIDATION")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("\nTEST")
print("X:", X_test.shape)
print("y:", y_test.shape)

FEATURE / TARGET SHAPES

TRAIN
X: (314982, 12)
y: (314982,)

VALIDATION
X: (39402, 12)
y: (39402,)

TEST
X: (41471, 12)
y: (41471,)


In [27]:
# ============================================
# STEP 23 — Separate categorical feature types
# ============================================

text_categorical_features = [
    "GENDER",
    "ABO",
    "ON_DIALYSIS",
    "A2A2B_status"
]

coded_categorical_features = [
    "ETHCAT",
    "FUNC_STAT_TCR_clean",
    "INIT_STAT",
    "REGION"
]

print("TEXT CATEGORICAL FEATURES")
for col in text_categorical_features:
    print("✓", col)

print("\nCODED CATEGORICAL FEATURES")
for col in coded_categorical_features:
    print("✓", col)

print(
    "\nTotal categorical features:",
    len(text_categorical_features)
    + len(coded_categorical_features)
)

TEXT CATEGORICAL FEATURES
✓ GENDER
✓ ABO
✓ ON_DIALYSIS
✓ A2A2B_status

CODED CATEGORICAL FEATURES
✓ ETHCAT
✓ FUNC_STAT_TCR_clean
✓ INIT_STAT
✓ REGION

Total categorical features: 8


In [28]:
# ============================================
# STEP 24 — Build preprocessing pipeline
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


# --------------------------------------------
# Numerical preprocessing
# Median imputation + scaling
# --------------------------------------------
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


# --------------------------------------------
# Text categorical preprocessing
# Missing category + one-hot encoding
# --------------------------------------------
text_categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])


# --------------------------------------------
# Numeric-coded categorical preprocessing
# Missing category + one-hot encoding
# --------------------------------------------
coded_categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=-999
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])


# --------------------------------------------
# Combine all preprocessing rules
# --------------------------------------------
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "text_categorical",
        text_categorical_pipeline,
        text_categorical_features
    ),
    (
        "coded_categorical",
        coded_categorical_pipeline,
        coded_categorical_features
    ),
    (
        "binary",
        "passthrough",
        binary_features
    )
])


print("PREPROCESSOR CREATED SUCCESSFULLY")

print("\nNumerical features:", numeric_features)
print("Text categorical:", text_categorical_features)
print("Coded categorical:", coded_categorical_features)
print("Binary features:", binary_features)

PREPROCESSOR CREATED SUCCESSFULLY

Numerical features: ['INIT_AGE', 'BMI_TCR_clean']
Text categorical: ['GENDER', 'ABO', 'ON_DIALYSIS', 'A2A2B_status']
Coded categorical: ['ETHCAT', 'FUNC_STAT_TCR_clean', 'INIT_STAT', 'REGION']
Binary features: ['BMI_missing', 'FUNC_STAT_missing']


In [29]:
# ============================================
# STEP 25 — Fit preprocessing on TRAIN only
# ============================================

# Learn preprocessing rules from training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same learned rules to validation/test
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)


print("PREPROCESSING COMPLETE")

print("\nTRAIN processed shape:")
print(X_train_processed.shape)

print("\nVALIDATION processed shape:")
print(X_val_processed.shape)

print("\nTEST processed shape:")
print(X_test_processed.shape)

PREPROCESSING COMPLETE

TRAIN processed shape:
(314982, 67)

VALIDATION processed shape:
(39402, 67)

TEST processed shape:
(41471, 67)


In [30]:
# ============================================
# STEP 26 — Validate processed feature matrices
# ============================================

import numpy as np

print("PROCESSED DATA VALIDATION")

print(
    "TRAIN missing values:",
    np.isnan(X_train_processed.data).sum()
)

print(
    "VALIDATION missing values:",
    np.isnan(X_val_processed.data).sum()
)

print(
    "TEST missing values:",
    np.isnan(X_test_processed.data).sum()
)

print("\nNumber of processed features:")
print(X_train_processed.shape[1])

PROCESSED DATA VALIDATION
TRAIN missing values: 0
VALIDATION missing values: 0
TEST missing values: 0

Number of processed features:
67


In [31]:
# ============================================
# STEP 27 — Inspect processed feature names
# ============================================

processed_feature_names = (
    preprocessor.get_feature_names_out()
)

print("TOTAL PROCESSED FEATURES:")
print(len(processed_feature_names))

print("\nFIRST 30 PROCESSED FEATURES:")

for feature in processed_feature_names[:30]:
    print("✓", feature)

TOTAL PROCESSED FEATURES:
67

FIRST 30 PROCESSED FEATURES:
✓ numeric__INIT_AGE
✓ numeric__BMI_TCR_clean
✓ text_categorical__GENDER_F
✓ text_categorical__GENDER_M
✓ text_categorical__ABO_A
✓ text_categorical__ABO_A1
✓ text_categorical__ABO_A1B
✓ text_categorical__ABO_A2
✓ text_categorical__ABO_A2B
✓ text_categorical__ABO_AB
✓ text_categorical__ABO_B
✓ text_categorical__ABO_O
✓ text_categorical__ON_DIALYSIS_N
✓ text_categorical__ON_DIALYSIS_Y
✓ text_categorical__A2A2B_status_B - missing
✓ text_categorical__A2A2B_status_B - recorded 0
✓ text_categorical__A2A2B_status_B - recorded 1
✓ text_categorical__A2A2B_status_Not applicable (non-B)
✓ coded_categorical__ETHCAT_1.0
✓ coded_categorical__ETHCAT_2.0
✓ coded_categorical__ETHCAT_4.0
✓ coded_categorical__ETHCAT_5.0
✓ coded_categorical__ETHCAT_6.0
✓ coded_categorical__ETHCAT_7.0
✓ coded_categorical__ETHCAT_9.0
✓ coded_categorical__ETHCAT_998.0
✓ coded_categorical__FUNC_STAT_TCR_clean_-999.0
✓ coded_categorical__FUNC_STAT_TCR_clean_996.0
✓ cod

In [32]:
# ============================================
# STEP 28 — Define CPRA sensitivity
# preprocessing groups
# ============================================

numeric_features_with_cpra = [
    "INIT_AGE",
    "BMI_TCR_clean",
    "INIT_CPRA"
]

binary_features_with_cpra = [
    "BMI_missing",
    "FUNC_STAT_missing",
    "INIT_CPRA_missing"
]

print("CPRA SENSITIVITY PREPROCESSING")

print("\nNumerical features:")
for col in numeric_features_with_cpra:
    print("✓", col)

print("\nCategorical features:")
for col in (
    text_categorical_features
    + coded_categorical_features
):
    print("✓", col)

print("\nBinary features:")
for col in binary_features_with_cpra:
    print("✓", col)

print(
    "\nTotal original features:",
    len(numeric_features_with_cpra)
    + len(text_categorical_features)
    + len(coded_categorical_features)
    + len(binary_features_with_cpra)
)

CPRA SENSITIVITY PREPROCESSING

Numerical features:
✓ INIT_AGE
✓ BMI_TCR_clean
✓ INIT_CPRA

Categorical features:
✓ GENDER
✓ ABO
✓ ON_DIALYSIS
✓ A2A2B_status
✓ ETHCAT
✓ FUNC_STAT_TCR_clean
✓ INIT_STAT
✓ REGION

Binary features:
✓ BMI_missing
✓ FUNC_STAT_missing
✓ INIT_CPRA_missing

Total original features: 14


In [33]:
# ============================================
# STEP 29 — Build CPRA sensitivity preprocessor
# ============================================

preprocessor_with_cpra = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features_with_cpra
    ),
    (
        "text_categorical",
        text_categorical_pipeline,
        text_categorical_features
    ),
    (
        "coded_categorical",
        coded_categorical_pipeline,
        coded_categorical_features
    ),
    (
        "binary",
        "passthrough",
        binary_features_with_cpra
    )
])

print("CPRA SENSITIVITY PREPROCESSOR CREATED")

print("\nNumerical:")
print(numeric_features_with_cpra)

print("\nBinary:")
print(binary_features_with_cpra)

print("\nTotal original features:")
print(len(features_with_cpra))

CPRA SENSITIVITY PREPROCESSOR CREATED

Numerical:
['INIT_AGE', 'BMI_TCR_clean', 'INIT_CPRA']

Binary:
['BMI_missing', 'FUNC_STAT_missing', 'INIT_CPRA_missing']

Total original features:
14


In [34]:
# ============================================
# STEP 30 — Process CPRA sensitivity features
# ============================================

# Create feature sets WITH CPRA
X_train_cpra = train_df[features_with_cpra].copy()
X_val_cpra = val_df[features_with_cpra].copy()
X_test_cpra = test_df[features_with_cpra].copy()


# Fit using TRAIN only
X_train_cpra_processed = (
    preprocessor_with_cpra.fit_transform(X_train_cpra)
)

# Apply training-learned rules to validation/test
X_val_cpra_processed = (
    preprocessor_with_cpra.transform(X_val_cpra)
)

X_test_cpra_processed = (
    preprocessor_with_cpra.transform(X_test_cpra)
)


print("CPRA SENSITIVITY PREPROCESSING COMPLETE")

print("\nTRAIN:")
print(X_train_cpra_processed.shape)

print("\nVALIDATION:")
print(X_val_cpra_processed.shape)

print("\nTEST:")
print(X_test_cpra_processed.shape)

CPRA SENSITIVITY PREPROCESSING COMPLETE

TRAIN:
(314982, 69)

VALIDATION:
(39402, 69)

TEST:
(41471, 69)


In [35]:
# ============================================
# STEP 31 — Validate CPRA processed matrices
# ============================================

print("CPRA PROCESSED DATA VALIDATION")

print(
    "TRAIN missing values:",
    np.isnan(X_train_cpra_processed.data).sum()
)

print(
    "VALIDATION missing values:",
    np.isnan(X_val_cpra_processed.data).sum()
)

print(
    "TEST missing values:",
    np.isnan(X_test_cpra_processed.data).sum()
)

print(
    "\nNumber of processed features:",
    X_train_cpra_processed.shape[1]
)

CPRA PROCESSED DATA VALIDATION
TRAIN missing values: 0
VALIDATION missing values: 0
TEST missing values: 0

Number of processed features: 69


In [36]:
# ============================================
# STEP 32 — Save model preparation outputs
# ============================================

from scipy import sparse
import joblib
import numpy as np

# --------------------------------------------
# PRIMARY MODEL — NO CPRA
# --------------------------------------------

sparse.save_npz(
    "/content/X_train_no_cpra.npz",
    X_train_processed
)

sparse.save_npz(
    "/content/X_val_no_cpra.npz",
    X_val_processed
)

sparse.save_npz(
    "/content/X_test_no_cpra.npz",
    X_test_processed
)


# --------------------------------------------
# CPRA SENSITIVITY VERSION
# --------------------------------------------

sparse.save_npz(
    "/content/X_train_with_cpra.npz",
    X_train_cpra_processed
)

sparse.save_npz(
    "/content/X_val_with_cpra.npz",
    X_val_cpra_processed
)

sparse.save_npz(
    "/content/X_test_with_cpra.npz",
    X_test_cpra_processed
)


# --------------------------------------------
# TARGETS
# --------------------------------------------

np.save("/content/y_train.npy", y_train.to_numpy())
np.save("/content/y_val.npy", y_val.to_numpy())
np.save("/content/y_test.npy", y_test.to_numpy())


# --------------------------------------------
# PREPROCESSORS
# --------------------------------------------

joblib.dump(
    preprocessor,
    "/content/preprocessor_no_cpra.joblib"
)

joblib.dump(
    preprocessor_with_cpra,
    "/content/preprocessor_with_cpra.joblib"
)


# --------------------------------------------
# FEATURE NAMES
# --------------------------------------------

no_cpra_feature_names = (
    preprocessor.get_feature_names_out()
)

with_cpra_feature_names = (
    preprocessor_with_cpra.get_feature_names_out()
)

pd.DataFrame({
    "feature": no_cpra_feature_names
}).to_csv(
    "/content/features_no_cpra.csv",
    index=False
)

pd.DataFrame({
    "feature": with_cpra_feature_names
}).to_csv(
    "/content/features_with_cpra.csv",
    index=False
)


print("MODEL PREPARATION FILES SAVED")

print("\nPrimary model:")
print("✓ X_train_no_cpra.npz")
print("✓ X_val_no_cpra.npz")
print("✓ X_test_no_cpra.npz")

print("\nCPRA sensitivity:")
print("✓ X_train_with_cpra.npz")
print("✓ X_val_with_cpra.npz")
print("✓ X_test_with_cpra.npz")

print("\nTargets:")
print("✓ y_train.npy")
print("✓ y_val.npy")
print("✓ y_test.npy")

print("\nPreprocessors + feature names saved.")

MODEL PREPARATION FILES SAVED

Primary model:
✓ X_train_no_cpra.npz
✓ X_val_no_cpra.npz
✓ X_test_no_cpra.npz

CPRA sensitivity:
✓ X_train_with_cpra.npz
✓ X_val_with_cpra.npz
✓ X_test_with_cpra.npz

Targets:
✓ y_train.npy
✓ y_val.npy
✓ y_test.npy

Preprocessors + feature names saved.


In [37]:
# ============================================
# STEP 33 — Save train/validation/test metadata
# ============================================

metadata_to_save = [
    "PT_CODE",
    "WL_ID_CODE",
    "INIT_DATE",
    "listing_year",
    "LISTING_CTR_CODE",
    "INIT_AGE",
    "GENDER",
    "ETHCAT",
    "ABO",
    "REGION",
    "adverse_1yr"
]


train_metadata = train_df[
    metadata_to_save
].copy()

val_metadata = val_df[
    metadata_to_save
].copy()

test_metadata = test_df[
    metadata_to_save
].copy()


train_metadata.to_csv(
    "/content/train_metadata.csv.gz",
    index=False,
    compression="gzip"
)

val_metadata.to_csv(
    "/content/val_metadata.csv.gz",
    index=False,
    compression="gzip"
)

test_metadata.to_csv(
    "/content/test_metadata.csv.gz",
    index=False,
    compression="gzip"
)


print("SPLIT METADATA SAVED")

print("\nTRAIN:", train_metadata.shape)
print("VALIDATION:", val_metadata.shape)
print("TEST:", test_metadata.shape)

print("\nRow alignment check:")
print(
    "Train:",
    len(train_metadata) == X_train_processed.shape[0]
)

print(
    "Validation:",
    len(val_metadata) == X_val_processed.shape[0]
)

print(
    "Test:",
    len(test_metadata) == X_test_processed.shape[0]
)

SPLIT METADATA SAVED

TRAIN: (314982, 11)
VALIDATION: (39402, 11)
TEST: (41471, 11)

Row alignment check:
Train: True
Validation: True
Test: True


In [38]:
# ============================================
# STEP 34 — Package model preparation outputs
# ============================================

import zipfile
import os

files_to_package = [
    # Primary no-CPRA matrices
    "/content/X_train_no_cpra.npz",
    "/content/X_val_no_cpra.npz",
    "/content/X_test_no_cpra.npz",

    # CPRA sensitivity matrices
    "/content/X_train_with_cpra.npz",
    "/content/X_val_with_cpra.npz",
    "/content/X_test_with_cpra.npz",

    # Targets
    "/content/y_train.npy",
    "/content/y_val.npy",
    "/content/y_test.npy",

    # Preprocessors
    "/content/preprocessor_no_cpra.joblib",
    "/content/preprocessor_with_cpra.joblib",

    # Feature names
    "/content/features_no_cpra.csv",
    "/content/features_with_cpra.csv",

    # Metadata
    "/content/train_metadata.csv.gz",
    "/content/val_metadata.csv.gz",
    "/content/test_metadata.csv.gz"
]

zip_path = "/content/medimate_model_preparation_outputs.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file_path in files_to_package:

        zipf.write(
            file_path,
            arcname=os.path.basename(file_path)
        )


print("MODEL PREPARATION PACKAGE CREATED")
print("File:", zip_path)

print("\nFiles inside package:")
for file_path in files_to_package:
    print("✓", os.path.basename(file_path))

print("\nTotal files:", len(files_to_package))

MODEL PREPARATION PACKAGE CREATED
File: /content/medimate_model_preparation_outputs.zip

Files inside package:
✓ X_train_no_cpra.npz
✓ X_val_no_cpra.npz
✓ X_test_no_cpra.npz
✓ X_train_with_cpra.npz
✓ X_val_with_cpra.npz
✓ X_test_with_cpra.npz
✓ y_train.npy
✓ y_val.npy
✓ y_test.npy
✓ preprocessor_no_cpra.joblib
✓ preprocessor_with_cpra.joblib
✓ features_no_cpra.csv
✓ features_with_cpra.csv
✓ train_metadata.csv.gz
✓ val_metadata.csv.gz
✓ test_metadata.csv.gz

Total files: 16
